### Parte 2 · XML de la administración

Usa licitaciones_pcsp.xml, un extracto real del canal Atom de la Plataforma de Contratación del Sector Público. Lleva espacios de nombres, que es la dificultad de la parte.

1. Ábrelo en un editor de texto y míralo antes de programar nada. Identifica la raíz, los espacios de nombres declarados y la jerarquía.


2. Explica en el cuaderno qué aporta cada etiqueta que vas a usar y en qué se diferencia una etiqueta de un atributo. Pon al menos un ejemplo de cada uno del propio documento.


3. Parsea con xml.etree.ElementTree gestionando los espacios de nombres correctamente (diccionario de prefijos en findall). Extrae los mismos cinco campos que en la parte 1.


4. Explica la diferencia entre que este documento esté bien formado y que sea válido, y di qué haría falta para comprobar lo segundo.


### 2. Etiquetas usadas y diferencia etiqueta / atributo

#### 2.1. Qué aporta cada etiqueta que se va a usar

Para extraer los cinco campos de la parte 1 (`id`, `organo`, `objeto`, `importe`, `fechaLimite`) se recorren estas etiquetas:

| Etiqueta | Namespace | Aporta | Nivel |
|---|---|---|---|
| `feed` | Atom | Raíz del documento; agrupa todos los anuncios del canal | raíz |
| `entry` | Atom | Cada licitación publicada (un bloque completo de datos) | hijo de `feed` |
| `id` | Atom | Identificador global del anuncio, p. ej. `urn:ejemplo:licitacion:2026-XML-0101` | hijo de `entry` |
| `title` | Atom | Título del anuncio; coincide con el objeto del contrato | hijo de `entry` |
| `ContractFolderID` | `cbc` | Identificador del expediente en formato corto: `2026-XML-0101` (es el `id` que usamos, para mantener el formato de la parte 1) | dentro de `cac:ContractFolderStatus` |
| `ContractFolderStatusCode` | `cbc` | Estado del expediente (`PUB` = publicado) | dentro de `cac:ContractFolderStatus` |
| `LocatedContractingParty` | `cac` | Contenedor del órgano que contrata | dentro de `cac:ContractFolderStatus` |
| `Party` | `cac` | Datos de la parte contratante | dentro de `LocatedContractingParty` |
| `PartyName` | `cac` | Nombre de la parte | dentro de `Party` |
| `Name` (en `PartyName`) | `cbc` | **Texto del órgano de contratación** | dentro de `PartyName` |
| `ProcurementProject` | `cac` | Contenedor del objeto y del presupuesto del contrato | dentro de `cac:ContractFolderStatus` |
| `Name` (en `ProcurementProject`) | `cbc` | **Objeto del contrato** (duplica `atom:title`) | dentro de `ProcurementProject` |
| `BudgetAmount` | `cac` | Contenedor del importe | dentro de `ProcurementProject` |
| `TotalAmount` | `cbc` | **Importe total** como texto (`1245000.00`); la moneda va en el atributo `currencyID` | dentro de `BudgetAmount` |
| `TenderingProcess` | `cac` | Contenedor del proceso de licitación | dentro de `cac:ContractFolderStatus` |
| `TenderSubmissionDeadlinePeriod` | `cac` | Periodo de presentación de ofertas | dentro de `TenderingProcess` |
| `EndDate` | `cbc` | **Fecha límite, parte fecha** (`2026-10-20`) | dentro de `TenderSubmissionDeadlinePeriod` |
| `EndTime` | `cbc` | **Fecha límite, parte hora** (`14:00:00`) | dentro de `TenderSubmissionDeadlinePeriod` |

Resumen de lo que aporta cada bloque:

- **`feed` y `entry`** dan la **estructura**: `feed` es el documento, `entry` es cada licitación. Sin `entry` no hay unidad de registro.
- **`id` / `ContractFolderID`** dan la **identidad** del expediente.
- **`title` / `Name`** dan el **objeto** del contrato (uno en Atom, otro en CODICE; coinciden).
- **`Name` bajo `PartyName`** da el **órgano de contratación**.
- **`TotalAmount`** da el **importe** (en `.text`), y su atributo `currencyID` da la **moneda**.
- **`EndDate` + `EndTime`** dan conjuntamente la **fecha límite**: hay que concatenarlas (`2026-10-20T14:00:00`) antes de pasarlas a `datetime`.

#### 2.2. Diferencia entre etiqueta y atributo

- **Etiqueta (elemento)**: nodo con nombre que puede contener **texto** y **otros elementos hijos**. En `ElementTree` se accede a su contenido con `.text` y a sus hijos con `find`/`findall`.
- **Atributo**: par `nombre="valor"` dentro de la etiqueta de apertura. **No tiene hijos ni contenido propio**; describe o califica al elemento. Se accede con `.get("nombre")` o `.attrib`.

En una frase: la etiqueta **es** el dato (o lo contiene); el atributo **califica** ese dato (moneda, idioma, URL, estado…).



### 3. Parseo con `xml.etree.ElementTree` y extracción de los cinco campos

#### 3.1. Namespaces y diccionario de prefijos

`ElementTree` no acepta prefijos tipo `cac:Name` en `findall`. Hay dos vías:

1. Usar el **tag expandido** con la URI entre llaves: `"{urn:...}Name"`.
2. Pasar un **diccionario de prefijos** como segundo argumento de `find`/`findall`; `ElementTree` sustituye cada `cac:`, `cbc:`, `atom:` por su URI.

La segunda es la que pide el ejercicio:

```python
import xml.etree.ElementTree as ET
from datetime import datetime

NS = {
    "atom": "http://www.w3.org/2005/Atom",
    "cbc":  "urn:dgpe:names:draft:codice:schema:xsd:CommonBasicComponents-2",
    "cac":  "urn:dgpe:names:draft:codice:schema:xsd:CommonAggregateComponents-2",
}

### 4. Bien formado vs válido

- **Bien formado**: cumple la sintaxis XML (un solo raíz, etiquetas bien cerradas/anidadas, atributos entrecomillados, namespaces declarados). Lo comprueba `ET.parse()`; si falla, lanza `ParseError`. **Este documento sí lo está.**
- **Válido**: además cumple un **esquema** (XSD/DTD): orden, cardinalidad, atributos obligatorios y tipos. **No se puede saber sin el esquema oficial.**

| | Bien formado | Válido |
|---|---|---|
| Comprueba | Sintaxis XML | Conformidad con un esquema |
| Herramienta | `ET.parse()` | Validador + esquema |
| Este fichero | Sí | No verificable sin esquema |

**Para comprobar la validez**: tener los XSD oficiales (Atom y CODICE/PCSP) y validar con `lxml` (`etree.XMLSchema(...).validate(doc)`) o `xmllint --schema`. `ElementTree` no valida.